# Dixon-Coles Model

Extends Poisson GLM with the Dixon-Coles ρ correction for low-score scorelines.
The ρ parameter adjusts the joint probability of (0-0, 1-0, 0-1, 1-1) outcomes,
which are systematically mis-predicted by independent Poisson models.

Key change vs Poisson GLM:
- Joint goal distribution: `P(h,a) = τ(h,a,λh,λa,ρ) · Poisson(h|λh) · Poisson(a|λa)`
- τ is the Dixon-Coles correction factor (=1 outside the 2×2 low-score block)
- ρ is estimated by MLE per fold

Evaluation: LOTO-CV over WC 2010/2014/2018/2022 (256 pooled matches).
Score prediction: maximise expected Sporza pts over the corrected joint distribution.

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from functools import lru_cache
from scipy.stats import poisson
from scipy.optimize import minimize_scalar
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

HOME_FEATURES = ['home_elo', 'away_elo', 'elo_diff', 'is_neutral',
                 'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
                 'home_form_ga', 'away_form_ga', 'tournament_weight']
AWAY_FEATURES = ['away_elo', 'home_elo', 'elo_diff', 'is_neutral',
                 'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
                 'away_form_ga', 'home_form_ga', 'tournament_weight']

In [2]:
def tau(home_g: int, away_g: int, lh: float, la: float, rho: float) -> float:
    """Dixon-Coles correction factor for low-score outcomes."""
    if home_g == 0 and away_g == 0:
        return 1 - lh * la * rho
    if home_g == 1 and away_g == 0:
        return 1 + la * rho
    if home_g == 0 and away_g == 1:
        return 1 + lh * rho
    if home_g == 1 and away_g == 1:
        return 1 - rho
    return 1.0


def dc_joint_pmf(lh: float, la: float, rho: float, max_goals: int = MAX_GOALS) -> np.ndarray:
    """Full (max_goals+1) x (max_goals+1) joint PMF under Dixon-Coles."""
    goals = np.arange(max_goals + 1)
    ph = poisson.pmf(goals, lh)
    pa = poisson.pmf(goals, la)
    joint = np.outer(ph, pa)
    # Apply τ correction to the 2x2 low-score block
    for h in range(2):
        for a in range(2):
            joint[h, a] *= tau(h, a, lh, la, rho)
    return joint


def neg_log_likelihood(rho: float, lambdas_h: np.ndarray, lambdas_a: np.ndarray,
                       actual_h: np.ndarray, actual_a: np.ndarray) -> float:
    """Negative log-likelihood for ρ given predicted λs and observed scores."""
    nll = 0.0
    for lh, la, ah, aa in zip(lambdas_h, lambdas_a, actual_h, actual_a):
        t = tau(int(ah), int(aa), lh, la, rho)
        # Guard against invalid τ values (must be positive for valid probability)
        if t <= 0:
            return 1e12
        p_h = poisson.pmf(int(ah), lh)
        p_a = poisson.pmf(int(aa), la)
        nll -= np.log(max(t * p_h * p_a, 1e-12))
    return nll


def estimate_rho(lambdas_h: np.ndarray, lambdas_a: np.ndarray,
                 actual_h: np.ndarray, actual_a: np.ndarray) -> float:
    """MLE estimate of ρ via bounded scalar search."""
    # ρ must satisfy τ > 0 for all low-score cells; practical range is (-1, 1)
    result = minimize_scalar(
        neg_log_likelihood,
        bounds=(-0.99, 0.99),
        method='bounded',
        args=(lambdas_h, lambdas_a, actual_h, actual_a),
    )
    return float(result.x)

In [3]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def build_X(df: pd.DataFrame, features: list) -> np.ndarray:
    X = df[features].copy().fillna(df[features].median())
    return X.values


def build_pipe() -> Pipeline:
    return Pipeline([('scaler', StandardScaler()),
                     ('poisson', PoissonRegressor(alpha=0.1, max_iter=300))])


def best_score_dc(lh: float, la: float, rho: float) -> tuple:
    """(pred_h, pred_a) maximising expected Sporza pts under Dixon-Coles joint."""
    joint = dc_joint_pmf(lh, la, rho)

    def expected_sporza(ph: int, pa: int) -> float:
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if ph == ah and pa == aa:
                    pts += p * 10
                elif (ph - pa) == (ah - aa):
                    pts += p * 7
                elif np.sign(ph - pa) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph in range(MAX_GOALS + 1):
        for pa in range(MAX_GOALS + 1):
            ep = expected_sporza(ph, pa)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph, pa
    return best_h, best_a

In [4]:
import json
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline (measured): {autofill_mean:.3f} pts/match')

Autofill baseline (measured): 4.137 pts/match


In [5]:
all_pts = []
fold_results = []
fold_rhos = {}

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    # Step 1: fit Poisson GLM on training data
    X_h = build_X(train, HOME_FEATURES)
    X_a = build_X(train, AWAY_FEATURES)
    X_tr = np.vstack([X_h, X_a])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

    pipe = build_pipe()
    pipe.fit(X_tr, y_tr)

    # Step 2: estimate ρ via MLE on training data
    lh_train = pipe.predict(build_X(train, HOME_FEATURES))
    la_train = pipe.predict(build_X(train, AWAY_FEATURES))
    rho = estimate_rho(lh_train, la_train,
                       train['home_score'].values, train['away_score'].values)
    fold_rhos[year] = rho
    print(f'WC {year} — estimated ρ = {rho:.4f}')

    # Step 3: predict on eval fold using Dixon-Coles joint
    lh_eval = pipe.predict(build_X(evalf, HOME_FEATURES))
    la_eval = pipe.predict(build_X(evalf, AWAY_FEATURES))

    preds = [best_score_dc(float(lh), float(la), rho)
             for lh, la in zip(lh_eval, la_eval)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(pred_home, pred_away,
                         evalf['home_score'].reset_index(drop=True),
                         evalf['away_score'].reset_index(drop=True))
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({'year': year, 'rho': rho, 'mean_pts': bd['mean_pts'],
                         'pct_exact': bd['pct_exact'],
                         'pct_correct_result': bd['pct_correct_result']})
    print(f'         mean_pts={bd["mean_pts"]:.3f}  exact={bd["pct_exact"]:.1%}  '
          f'correct={bd["pct_correct_result"]:.1%}')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
print(f'\nPooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  '
      f'CI width={ci["ci_hi"]-ci["ci_lo"]:.3f}')

WC 2010 — estimated ρ = -0.0159


         mean_pts=4.219  exact=18.8%  correct=51.6%


WC 2014 — estimated ρ = -0.0421
         mean_pts=4.547  exact=14.1%  correct=60.9%


WC 2018 — estimated ρ = -0.0368
         mean_pts=4.703  exact=20.3%  correct=59.4%


WC 2022 — estimated ρ = -0.0256
         mean_pts=3.844  exact=9.4%  correct=54.7%

Pooled LOTO-CV (256 matches): 4.328 pts/match  95% CI [3.926, 4.734]  CI width=0.809


In [6]:
# Comparison vs baselines
poisson_glm_mean = 4.336  # from notebook 11

print('Comparison vs baselines:')
print(f'  Autofill:             {autofill_mean:.3f} pts/match')
print(f'  Poisson GLM LOTO-CV:  {poisson_glm_mean:.3f} pts/match')
print(f'  Dixon-Coles LOTO-CV:  {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Δ vs autofill:        {ci["mean"] - autofill_mean:+.3f} pts/match')
print(f'  Δ vs Poisson GLM:     {ci["mean"] - poisson_glm_mean:+.3f} pts/match')

beats_autofill = ci['ci_lo'] > autofill_mean
print(f'  CI lower > autofill:  {beats_autofill} → {"statistically beats" if beats_autofill else "not conclusive"}')

print('\nPer-fold ρ values:')
for r in fold_results:
    print(f'  WC {r["year"]}: ρ={r["rho"]:+.4f}  mean_pts={r["mean_pts"]:.3f}  '
          f'exact={r["pct_exact"]:.1%}  correct={r["pct_correct_result"]:.1%}')

Comparison vs baselines:
  Autofill:             4.137 pts/match
  Poisson GLM LOTO-CV:  4.336 pts/match
  Dixon-Coles LOTO-CV:  4.328 pts/match  95% CI [3.926, 4.734]
  Δ vs autofill:        +0.191 pts/match
  Δ vs Poisson GLM:     -0.008 pts/match
  CI lower > autofill:  False → not conclusive

Per-fold ρ values:
  WC 2010: ρ=-0.0159  mean_pts=4.219  exact=18.8%  correct=51.6%
  WC 2014: ρ=-0.0421  mean_pts=4.547  exact=14.1%  correct=60.9%
  WC 2018: ρ=-0.0368  mean_pts=4.703  exact=20.3%  correct=59.4%
  WC 2022: ρ=-0.0256  mean_pts=3.844  exact=9.4%  correct=54.7%


In [7]:
mlflow.set_tracking_uri('../../mlruns')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='dixon_coles_loto'):
    mlflow.set_tags({'modality': 'tabular', 'approach': 'dixon_coles',
                     'eval_strategy': 'loto_cv', 'dataset_version': 'split_v2'})
    mlflow.log_params({'features': ','.join(HOME_FEATURES), 'alpha': 0.1,
                       'score_strategy': 'max_expected_sporza_pts',
                       'max_goals_grid': MAX_GOALS,
                       'rho_estimation': 'mle_per_fold'})
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_poisson_glm', ci['mean'] - poisson_glm_mean)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
        mlflow.log_metric(f'fold_{r["year"]}_rho', r['rho'])
    run_id = mlflow.active_run().info.run_id

print(f'run_id: {run_id}')

run_id: e30c2f2acf8448a08f218b7aeb6b5298
